# Structure Data Manager

Notebook điều phối crawler vnstock (module `structure_data`): retry/fallback đa nguồn, QC OHLCV, pipeline một lệnh tùy chọn.

**Đồ án:** Hệ thống Data Pipeline và tra cứu phân tích thị trường chứng khoán Việt Nam đa nguồn.
**Nguồn:** `primary_source` / `fallback_source` (vd. **VCI** + **KBS**). Key vnstock: tạo file **`stock-pipeline/.env`** với dòng `VNSTOCK_API_KEY=...` (đã có trong `.gitignore`); `register_vnstock_api_key_from_env` tự gọi `load_dotenv`. Có thể dùng `.env.example` làm mẫu.

**Mục đích:** 
- **Giá OHLCV (`price`)**, **chỉ số (`index`)**, **financial ratio (`financial_ratio`)**: lưu theo **`date=<ngày chạy notebook>`**. Mỗi mã (hoặc mỗi mã chỉ số) một file Parquet trong partition đó. **Chưa có** partition/file → tạo mới; **đã có** → **ghi đè**.
- **Company overview**: file master **`company/master/company_overview.parquet`** — cơ chế **append** (mỗi lần chạy thêm batch raw-ish kèm `snapshot_date`, `fetched_at`, `source`), **không** xóa lịch sử các lần trước.
- **Listing**: snapshot master tại `listing/master/` — mỗi lần chạy **ghi đè** file listing hiện tại.

**Gợi ý:** chạy từng bước (ô dưới) hoặc `run_structure_ingestion_pipeline(cfg)` để nghỉ `delay_between_categories_sec` giữa các nhóm API.

In [5]:
import os
import sys
import warnings
import threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Suppress encoding errors từ subprocess threads
original_hook = threading.excepthook
def custom_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass  # Ignore encoding errors from threads
    else:
        original_hook(args)
threading.excepthook = custom_hook

# Suppress non-critical warnings
warnings.filterwarnings("ignore")

print("[OK] UTF-8 guard enabled for current kernel session")

[OK] UTF-8 guard enabled for current kernel session


In [7]:
import os
import importlib
from datetime import datetime
import structure_data as sd
import structure_data.common as sd_common
import structure_data.config as sd_config
import structure_data.price_ingestor as sd_price_ingestor
import structure_data.index_ingestor as sd_index_ingestor
import structure_data.stock_info_ingestor as sd_stock_info_ingestor
import structure_data.pipeline as sd_pipeline

# Reload theo thu tu: submodule -> package facade (tranh cache module cu trong notebook kernel)
for _m in (
    sd_common,
    sd_config,
    sd_price_ingestor,
    sd_index_ingestor,
    sd_stock_info_ingestor,
    sd_pipeline,
):
    importlib.reload(_m)
sd = importlib.reload(sd)

from structure_data import (
    IngestionConfig,
    configure_logging,
    register_vnstock_api_key_from_env,
    run_structure_ingestion_pipeline,
    ingest_prices,
    ingest_indices,
    ingest_listing,
    ingest_company_overview,
    ingest_financial_ratio,
    ingest_price_board,
)

register_vnstock_api_key_from_env("VNSTOCK_API_KEY")

configure_logging()
cfg = IngestionConfig()

# Backward-compat: neu kernel dang giu dataclass cu thi bo sung field incremental
if not hasattr(cfg, "min_ohlcv_rows_stock_incremental"):
    cfg.min_ohlcv_rows_stock_incremental = 5
if not hasattr(cfg, "min_ohlcv_rows_index_incremental"):
    cfg.min_ohlcv_rows_index_incremental = 5

# TEST trong Jupyter: moi lan run tao 1 partition raw rieng (mo phong 1 DAG run)
cfg.run_partition = datetime.now().strftime("%Y-%m-%dT%H%M%S")

# Quan ly danh sach ma co phieu ngay tai notebook (them/bot moi lan chay)
cfg.tickers = [
    # VN30 hien tai (20 ma)
    "ACB","BCM","BID","BVH","CTG","FPT","GAS","GVR","HDB","HPG",
    "MBB","MSN","MWG","PLX","POW","SAB","SHB","SSI","STB","TCB",
    # Them 30 ma da nganh
    "VCB","VHM","VIC","VJC","VNM","VPB","VRE","TPB","VIB","HVN",
    "REE","PNJ","GMD","DGC","DPM","DCM","HSG","KDC","QNS","SCS",
    "NVL","PDR","DXG","KDH","HDG","EVF","KBC","AGG","TCH","VPI",
]

# Co the sua bo chi so neu can
cfg.index_tickers = ["VNINDEX", "VN30", "HNXINDEX", "HNX30", "UPCOMINDEX"]

cfg.max_tickers_per_run = len(cfg.tickers)
cfg.rate_limit_rpm = 10
cfg.inter_request_delay_sec = 0.5

# cfg.primary_source = "vci"
# cfg.fallback_source = "kbs"

RUN_PROFILE = "backfill"  # "backfill" | "daily_incremental"

PROFILE_OVERRIDES = {
    "backfill": {
        "use_incremental_window": False,
        "incremental_window_days": 10,
        "bootstrap_full_history_if_missing": True,
        "full_bootstrap_once_then_incremental": False,
        "full_bootstrap_marker_file": "_full_bootstrap_done.json",
        "years_back": 5,
    },
    "daily_incremental": {
        "use_incremental_window": True,
        "incremental_window_days": 1,
        "bootstrap_full_history_if_missing": True,
        "full_bootstrap_once_then_incremental": False,
        "full_bootstrap_marker_file": "_full_bootstrap_done.json",
        "years_back": 5,
    },
}

if RUN_PROFILE not in PROFILE_OVERRIDES:
    raise ValueError(
        f"RUN_PROFILE='{RUN_PROFILE}' khong hop le. Chon 1 trong: {list(PROFILE_OVERRIDES)}"
    )

for _key, _value in PROFILE_OVERRIDES[RUN_PROFILE].items():
    setattr(cfg, _key, _value)

print("structure_data module:", sd.__file__)
print("config module:", sd_config.__file__)
print("price_ingestor module:", sd_price_ingestor.__file__)
print('has ingest_price_board:', hasattr(sd, 'ingest_price_board'))
print("RUN_PROFILE:", RUN_PROFILE)
print("fetch_mode:", "incremental-window" if cfg.use_incremental_window else "full-backfill")
print("incremental_window_days:", cfg.incremental_window_days)
print("qc stock full/inc:", cfg.min_ohlcv_rows_stock, cfg.min_ohlcv_rows_stock_incremental)
print("qc index full/inc:", cfg.min_ohlcv_rows_index, cfg.min_ohlcv_rows_index_incremental)
print("full_bootstrap_once_then_incremental:", cfg.full_bootstrap_once_then_incremental)
print('run_partition:', cfg.run_partition)
print(cfg)

2026-05-18 17:15:11 [INFO] Loaded environment variables from D:\WorkSpace\Đồ Án 2\stock-pipeline\.env


2026-05-18 17:15:13 [INFO] API key setup completed
2026-05-18 17:15:13 [INFO] Called register_user from VNSTOCK_API_KEY.


✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***6437
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
structure_data module: d:\WorkSpace\Đồ Án 2\stock-pipeline\ingestion\structure_data\__init__.py
config module: d:\WorkSpace\Đồ Án 2\stock-pipeline\ingestion\structure_data\config.py
price_ingestor module: d:\WorkSpace\Đồ Án 2\stock-pipeline\ingestion\structure_data\price_ingestor.py
has ingest_price_board: True
RUN_PROFILE: backfill
fetch_mode: full-backfill
incremental_window_days: 10
qc stock full/inc: 50 5
qc index full/inc: 100 5
full_bootstrap_once_then_incremental: False
run_partition: 2026-05-18T171513
IngestionConfig(tickers=['ACB', 'BCM', 'BID', 'BVH', 'CTG', 'FPT', 'GAS', 'GVR', 'HDB', 'H

In [3]:
# 1) OHLCV gia co phieu
price_outputs = ingest_prices(cfg)
len(price_outputs), list(price_outputs.items())[:3]

2026-05-18 13:57:03 [INFO] Loaded environment variables from D:\WorkSpace\Đồ Án 2\stock-pipeline\.env
2026-05-18 13:57:03 [INFO] [1/50] Fetching ACB (full_5y: 2021-05-18 -> 2021-12-31, qc_min_rows=50)...


2026-05-18 13:57:06 [INFO] ACB Bronze columns: ['time', 'open', 'high', 'low', 'close', 'volume', 'ticker', 'ingested_at', 'fetched_at', 'source', 'instrument_type']
2026-05-18 13:57:06 [INFO] ACB | src_used=KBS rows=162 min_date=2021-05-18 max_date=2021-12-31 pct_missing_close=0.00% pct_missing_value=100.00%
2026-05-18 13:57:07 [INFO] Saved 10 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price\year=2021\month=05\ACB.parquet
2026-05-18 13:57:07 [INFO] Saved 22 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price\year=2021\month=06\ACB.parquet
2026-05-18 13:57:07 [INFO] Saved 22 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price\year=2021\month=07\ACB.parquet
2026-05-18 13:57:07 [INFO] Saved 22 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price\year=2021\month=08\ACB.parquet
2026-05-18 13:57:07 [INFO] Saved 20 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_D

(50,
 [('ACB',
   ['D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\year=2021\\month=05\\ACB.parquet',
    'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\year=2021\\month=06\\ACB.parquet',
    'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\year=2021\\month=07\\ACB.parquet',
    'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\year=2021\\month=08\\ACB.parquet',
    'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\year=2021\\month=09\\ACB.parquet',
    'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\year=2021\\month=10\\ACB.parquet',
    'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\year=2021\\month=11\\ACB.parquet',
    'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\year=2021\\month=12\\ACB.parquet',
    'D:\\WorkSpace\\Đồ Án 2\\stoc

In [4]:
# 2) Chi so VNINDEX/VN30/HNX...
index_outputs = ingest_indices(cfg)
index_outputs

2026-05-18 14:30:34 [INFO] [1/5] Fetching index VNINDEX (full_5y: 2021-05-18 -> 2021-12-31, qc_min_rows=100)...
2026-05-18 14:30:39 [INFO] VNINDEX Bronze columns: ['time', 'open', 'high', 'low', 'close', 'volume', 'ticker', 'ingested_at', 'fetched_at', 'source', 'instrument_type']
2026-05-18 14:30:39 [INFO] VNINDEX | src_used=KBS rows=162 min_date=2021-05-18 max_date=2021-12-31 pct_missing_close=0.00% pct_missing_value=100.00%
2026-05-18 14:30:39 [INFO] Saved 10 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\index\year=2021\month=05\VNINDEX.parquet
2026-05-18 14:30:39 [INFO] Saved 22 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\index\year=2021\month=06\VNINDEX.parquet
2026-05-18 14:30:39 [INFO] Saved 22 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\index\year=2021\month=07\VNINDEX.parquet
2026-05-18 14:30:39 [INFO] Saved 22 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\index\year

{'VNINDEX': ['D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\year=2021\\month=05\\VNINDEX.parquet',
  'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\year=2021\\month=06\\VNINDEX.parquet',
  'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\year=2021\\month=07\\VNINDEX.parquet',
  'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\year=2021\\month=08\\VNINDEX.parquet',
  'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\year=2021\\month=09\\VNINDEX.parquet',
  'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\year=2021\\month=10\\VNINDEX.parquet',
  'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\year=2021\\month=11\\VNINDEX.parquet',
  'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\year=2021\\month=12\\VNINDEX.parquet',
  'D:\\WorkSpace\\Đồ 

In [5]:
# 3) Listing
listing_file = ingest_listing(cfg)
listing_file

2026-05-18 14:34:28 [INFO] symbols_by_exchange: 3253 rows from KBS
2026-05-18 14:34:28 [INFO] Saved raw-ish listing snapshot: 3253 rows -> D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet


'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\listing\\master\\listing.parquet'

In [6]:
# 4) Company overview (master append)
company_file = ingest_company_overview(cfg)
company_file

2026-05-18 14:34:28 [INFO] [1/50] Fetching company ACB...
2026-05-18 14:34:36 [INFO] [2/50] Fetching company BCM...
2026-05-18 14:34:42 [INFO] [3/50] Fetching company BID...
2026-05-18 14:34:46 [INFO] [4/50] Fetching company BVH...
2026-05-18 14:34:51 [INFO] [5/50] Fetching company CTG...
2026-05-18 14:34:57 [INFO] [6/50] Fetching company FPT...
2026-05-18 14:35:03 [INFO] [7/50] Fetching company GAS...
2026-05-18 14:35:10 [INFO] [8/50] Fetching company GVR...
2026-05-18 14:35:15 [INFO] [9/50] Fetching company HDB...
2026-05-18 14:35:21 [INFO] [10/50] Fetching company HPG...
2026-05-18 14:35:27 [INFO] [11/50] Fetching company MBB...
2026-05-18 14:35:33 [INFO] [12/50] Fetching company MSN...
2026-05-18 14:35:39 [INFO] [13/50] Fetching company MWG...
2026-05-18 14:35:45 [INFO] [14/50] Fetching company PLX...
2026-05-18 14:35:51 [INFO] [15/50] Fetching company POW...
2026-05-18 14:35:57 [INFO] [16/50] Fetching company SAB...
2026-05-18 14:36:03 [INFO] [17/50] Fetching company SHB...
2026-0

'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\company\\snapshots\\snapshot_date=2026-05-18T135703\\company_overview.parquet'

In [8]:
# 5) Financial ratio
ratio_outputs = ingest_financial_ratio(cfg)
len(ratio_outputs), list(ratio_outputs.items())[:3]

2026-05-18 17:15:35 [INFO] [1/50] Fetching financial ratio ACB...
2026-05-18 17:15:36 [INFO] Loaded 162 built-in KBS field mappings
2026-05-18 17:15:36 [INFO] Loaded 162 built-in KBS field mappings


2026-05-18 17:15:44 [INFO] Saved 32 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\financial_ratio\snapshot_date=2026-05-18T171513\ACB.parquet
2026-05-18 17:15:44 [INFO] Saved ratio ACB: 32 rows (src=kbs, period=quarter)
2026-05-18 17:15:44 [INFO] [2/50] Fetching financial ratio BCM...
2026-05-18 17:15:44 [INFO] Loaded 162 built-in KBS field mappings
2026-05-18 17:15:44 [INFO] Loaded 162 built-in KBS field mappings
2026-05-18 17:15:45 [INFO] Saved 58 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\financial_ratio\snapshot_date=2026-05-18T171513\BCM.parquet
2026-05-18 17:15:45 [INFO] Saved ratio BCM: 58 rows (src=kbs, period=quarter)
2026-05-18 17:15:45 [INFO] [3/50] Fetching financial ratio BID...
2026-05-18 17:15:50 [INFO] Loaded 162 built-in KBS field mappings
2026-05-18 17:15:50 [INFO] Loaded 162 built-in KBS field mappings
2026-05-18 17:15:51 [INFO] Saved 32 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\f

(50,
 [('ACB',
   'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\financial_ratio\\snapshot_date=2026-05-18T171513\\ACB.parquet'),
  ('BCM',
   'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\financial_ratio\\snapshot_date=2026-05-18T171513\\BCM.parquet'),
  ('BID',
   'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\financial_ratio\\snapshot_date=2026-05-18T171513\\BID.parquet')])

In [8]:
# 6) Price board snapshot
price_board_file = ingest_price_board(cfg)
price_board_file

2026-05-18 14:40:08 [INFO] Saved 50 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price_board\snapshot_at=2026-05-18T07-40-08\PRICE_BOARD_SNAPSHOT.parquet


'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price_board\\snapshot_at=2026-05-18T07-40-08\\PRICE_BOARD_SNAPSHOT.parquet'

In [ ]:
# 7) Tổng kết Structure Data (đúng layout Bronze: year/month, snapshot, master)
from __future__ import annotations

import json
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import pandas as pd

base = Path(cfg.data_lake_root)
EXPECTED_TICKERS = {
    str(t).strip().upper()
    for t in getattr(cfg, "tickers", [])[: getattr(cfg, "max_tickers_per_run", 50)]
    if str(t).strip()
}
EXPECTED_INDEXES = {
    str(t).strip().upper()
    for t in getattr(cfg, "index_tickers", [])
    if str(t).strip()
}

MONTHLY_OHLCV = {
    "price": ("OHLCV cổ phiếu", EXPECTED_TICKERS),
    "index": ("OHLCV chỉ số", EXPECTED_INDEXES),
}
SNAPSHOT_LAYOUTS = {
    "financial_ratio": ("snapshot_date", "Financial ratio"),
    "price_board": ("snapshot_at", "Price board"),
}
COMPANY_SNAPSHOT_ROOT = base / "company" / "snapshots"
MASTER_FILES = {
    "listing": (base / "listing" / "master" / "listing.parquet", "Listing master"),
}


def _parquet_rows(file_path: Path) -> int:
    try:
        import pyarrow.parquet as pq

        return int(pq.ParquetFile(file_path).metadata.num_rows)
    except Exception:
        try:
            return int(pd.read_parquet(file_path).shape[0])
        except Exception:
            return -1


def _human_size(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(max(int(num_bytes), 0))
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= 1024
    return f"{num_bytes} B"


def _path_partition_value(path: Path, key: str) -> str | None:
    prefix = f"{key}="
    for part in path.parts:
        if part.startswith(prefix):
            return part.split("=", 1)[1]
    return None


def _ym_from_monthly_path(path: Path) -> str | None:
    year = _path_partition_value(path, "year")
    month = _path_partition_value(path, "month")
    if not year or not month:
        return None
    return f"{year}-{int(month):02d}"


def _trading_range_from_parquet(path: Path) -> tuple[str | None, str | None]:
    try:
        import pyarrow.parquet as pq

        schema_names = {name.lower() for name in pq.ParquetFile(path).schema.names}
        cols = [c for c in ("trading_date", "time", "date") if c in schema_names]
        df = pd.read_parquet(path, columns=cols or None)
    except Exception:
        return None, None
    if df.empty:
        return None, None

    series = None
    if "trading_date" in df.columns:
        series = pd.to_datetime(df["trading_date"], errors="coerce")
    else:
        for col in ("time", "date"):
            if col in df.columns:
                series = pd.to_datetime(df[col], errors="coerce")
                break
    if series is None:
        return None, None
    series = series.dropna()
    if series.empty:
        return None, None
    return (
        pd.Timestamp(series.min()).date().isoformat(),
        pd.Timestamp(series.max()).date().isoformat(),
    )


def _scan_monthly_ohlcv(category: str, label: str, expected_symbols: set[str]) -> tuple[dict, pd.DataFrame]:
    root = base / category
    if not root.exists():
        return {
            "category": category,
            "label": label,
            "layout": "year=*/month=*/<TICKER>.parquet",
            "status": "MISSING",
            "file_count": 0,
            "month_partitions": 0,
            "symbol_count": 0,
            "expected_symbols": len(expected_symbols),
            "missing_symbols": ", ".join(sorted(expected_symbols)) if expected_symbols else "",
            "partition_from": "",
            "partition_to": "",
            "trading_date_from": "",
            "trading_date_to": "",
            "total_rows": 0,
            "total_size_human": _human_size(0),
        }, pd.DataFrame()

    files = sorted(root.glob("year=*/month=*/*.parquet"))
    by_symbol: dict[str, list[tuple[str, Path]]] = defaultdict(list)
    total_rows = 0
    total_bytes = 0
    ym_values: list[str] = []

    for file_path in files:
        symbol = file_path.stem.upper()
        ym = _ym_from_monthly_path(file_path)
        if ym:
            ym_values.append(ym)
            by_symbol[symbol].append((ym, file_path))
        rows = _parquet_rows(file_path)
        if rows >= 0:
            total_rows += rows
        try:
            total_bytes += int(file_path.stat().st_size)
        except OSError:
            pass

    symbol_rows: list[dict] = []
    trading_mins: list[str] = []
    trading_maxs: list[str] = []

    for symbol in sorted(by_symbol):
        items = sorted(by_symbol[symbol], key=lambda x: x[0])
        min_ym, min_file = items[0]
        max_ym, max_file = items[-1]
        td_min_a, td_max_a = _trading_range_from_parquet(min_file)
        td_min_b, td_max_b = _trading_range_from_parquet(max_file)
        td_candidates_min = [x for x in (td_min_a, td_min_b) if x]
        td_candidates_max = [x for x in (td_max_a, td_max_b) if x]
        td_min = min(td_candidates_min) if td_candidates_min else ""
        td_max = max(td_candidates_max) if td_candidates_max else ""
        if td_min:
            trading_mins.append(td_min)
        if td_max:
            trading_maxs.append(td_max)
        symbol_rows.append(
            {
                "category": category,
                "symbol": symbol,
                "month_files": len(items),
                "partition_from": min_ym,
                "partition_to": max_ym,
                "trading_date_from": td_min,
                "trading_date_to": td_max,
                "in_expected_batch": symbol in expected_symbols if expected_symbols else True,
            }
        )

    present = {row["symbol"] for row in symbol_rows}
    missing = sorted(expected_symbols - present) if expected_symbols else []
    summary = {
        "category": category,
        "label": label,
        "layout": "year=*/month=*/<TICKER>.parquet",
        "status": "OK" if files else "MISSING",
        "file_count": len(files),
        "month_partitions": len(set(ym_values)),
        "symbol_count": len(present),
        "expected_symbols": len(expected_symbols),
        "missing_symbols": ", ".join(missing),
        "partition_from": min(ym_values) if ym_values else "",
        "partition_to": max(ym_values) if ym_values else "",
        "trading_date_from": min(trading_mins) if trading_mins else "",
        "trading_date_to": max(trading_maxs) if trading_maxs else "",
        "total_rows": total_rows,
        "total_size_human": _human_size(total_bytes),
    }
    return summary, pd.DataFrame(symbol_rows)


def _scan_snapshot_category(category: str, partition_key: str, label: str) -> pd.DataFrame:
    root = base / category
    if not root.exists():
        return pd.DataFrame()

    rows: list[dict] = []
    for part_dir in sorted(root.glob(f"{partition_key}=*")):
        if not part_dir.is_dir():
            continue
        files = sorted(part_dir.glob("*.parquet"))
        total_rows = 0
        total_bytes = 0
        for file_path in files:
            rows_n = _parquet_rows(file_path)
            if rows_n >= 0:
                total_rows += rows_n
            try:
                total_bytes += int(file_path.stat().st_size)
            except OSError:
                pass
        rows.append(
            {
                "category": category,
                "label": label,
                "partition_key": partition_key,
                "partition": part_dir.name.split("=", 1)[-1],
                "file_count": len(files),
                "total_rows": total_rows,
                "total_size_human": _human_size(total_bytes),
                "status": "OK" if files else "EMPTY",
            }
        )
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["partition_sort"] = pd.to_datetime(out["partition"], errors="coerce")
    return out.sort_values("partition_sort", ascending=False, kind="stable").drop(
        columns="partition_sort"
    )


def _scan_company_snapshots() -> pd.DataFrame:
    if not COMPANY_SNAPSHOT_ROOT.exists():
        return pd.DataFrame()
    rows: list[dict] = []
    for part_dir in sorted(COMPANY_SNAPSHOT_ROOT.glob("snapshot_date=*")):
        file_path = part_dir / "company_overview.parquet"
        rows.append(
            {
                "category": "company",
                "label": "Company overview",
                "partition_key": "snapshot_date",
                "partition": part_dir.name.split("=", 1)[-1],
                "file_count": int(file_path.exists()),
                "total_rows": _parquet_rows(file_path) if file_path.exists() else 0,
                "total_size_human": _human_size(file_path.stat().st_size) if file_path.exists() else _human_size(0),
                "status": "OK" if file_path.exists() else "MISSING",
            }
        )
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["partition_sort"] = pd.to_datetime(out["partition"], errors="coerce")
    return out.sort_values("partition_sort", ascending=False, kind="stable").drop(
        columns="partition_sort"
    )


def _read_run_metadata() -> pd.DataFrame:
    rows: list[dict] = []
    for category in ("price", "index"):
        runs_dir = base / category / "_runs"
        if not runs_dir.exists():
            continue
        for meta_path in sorted(runs_dir.glob("*.json")):
            try:
                payload = json.loads(meta_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            rows.append(
                {
                    "category": category,
                    "run_id": payload.get("run_id", meta_path.stem),
                    "run_type": payload.get("run_type", ""),
                    "status": payload.get("status", ""),
                    "ingested_at": payload.get("ingested_at", ""),
                    "trading_date_from": payload.get("trading_date_from", ""),
                    "trading_date_to": payload.get("trading_date_to", ""),
                    "tickers_ok": len(payload.get("tickers") or []),
                    "row_count": payload.get("row_count", 0),
                }
            )
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["ingested_sort"] = pd.to_datetime(out["ingested_at"], errors="coerce", utc=True)
    return out.sort_values(["category", "ingested_sort"], ascending=[True, False], kind="stable").drop(
        columns="ingested_sort"
    )


def _session_ingest_hints() -> pd.DataFrame:
    rows = []
    checks = [
        ("price", "price_outputs", lambda o: len(o), lambda o: sum(len(v) for v in o.values())),
        ("index", "index_outputs", lambda o: len(o), lambda o: sum(len(v) for v in o.values())),
        ("listing", "listing_file", lambda o: 1 if o else 0, lambda o: 0),
        ("company", "company_file", lambda o: 1 if o else 0, lambda o: 0),
        ("financial_ratio", "ratio_outputs", lambda o: len(o), lambda o: 0),
        ("price_board", "price_board_file", lambda o: 1 if o else 0, lambda o: 0),
    ]
    g = globals()
    for category, var_name, sym_fn, file_fn in checks:
        if var_name not in g:
            continue
        value = g[var_name]
        if not value:
            continue
        rows.append(
            {
                "category": category,
                "session_var": var_name,
                "symbols_or_files": sym_fn(value),
                "parquet_paths": file_fn(value),
            }
        )
    return pd.DataFrame(rows)


# --- 0) Cấu hình run hiện tại ---
print("0) Cấu hình ingest (cfg)")
print(f"- data_lake_root: {base}")
print(f"- run_date (metadata): {cfg.run_date}")
print(f"- end_date API: {cfg.end_date}")
print(f"- tickers batch: {len(EXPECTED_TICKERS)} mã")
print(f"- index batch: {len(EXPECTED_INDEXES)} mã")
print(f"- incremental: {cfg.use_incremental_window} | window_days: {cfg.incremental_window_days}")
print(f"- years_back: {cfg.years_back} | sources: {cfg.resolved_data_sources()}")

watermark_path = base / "_watermark.json"
if watermark_path.exists():
    try:
        wm = json.loads(watermark_path.read_text(encoding="utf-8"))
        print(f"- watermark: {wm}")
    except Exception as ex:
        print(f"- watermark: không đọc được ({ex})")
else:
    print("- watermark: chưa có (_watermark.json — Silver cập nhật sau transform)")

for cat in ("price", "index"):
    marker = base / cat / "_full_bootstrap_done.json"
    if marker.exists():
        print(f"- bootstrap marker [{cat}]: có")

session_df = _session_ingest_hints()
if not session_df.empty:
    print("\nGợi ý từ cell vừa chạy trong kernel (nếu có):")
    display(session_df)

# --- A) OHLCV monthly ---
monthly_summaries: list[dict] = []
symbol_detail_frames: list[pd.DataFrame] = []
for category, (label, expected) in MONTHLY_OHLCV.items():
    summary, symbols_df = _scan_monthly_ohlcv(category, label, expected)
    monthly_summaries.append(summary)
    if not symbols_df.empty:
        symbol_detail_frames.append(symbols_df)

monthly_df = pd.DataFrame(monthly_summaries)
print("\nA) OHLCV — layout year=*/month=*/<TICKER>.parquet (merge/dedupe theo tháng)")
if monthly_df.empty:
    print("Chưa có dữ liệu price/index.")
else:
    display(
        monthly_df[
            [
                "category",
                "label",
                "status",
                "symbol_count",
                "expected_symbols",
                "missing_symbols",
                "file_count",
                "month_partitions",
                "partition_from",
                "partition_to",
                "trading_date_from",
                "trading_date_to",
                "total_rows",
                "total_size_human",
            ]
        ]
    )

if symbol_detail_frames:
    symbols_df = pd.concat(symbol_detail_frames, ignore_index=True)
    print("A.1) Chi tiết theo mã (partition tháng + trading_date min/max)")
    display(symbols_df)

# --- B) Snapshot datasets ---
snapshot_frames = []
for category, (partition_key, label) in SNAPSHOT_LAYOUTS.items():
    part_df = _scan_snapshot_category(category, partition_key, label)
    if not part_df.empty:
        snapshot_frames.append(part_df)

company_df = _scan_company_snapshots()
if not company_df.empty:
    snapshot_frames.append(company_df)

snapshot_df = pd.concat(snapshot_frames, ignore_index=True) if snapshot_frames else pd.DataFrame()
print("\nB) Snapshot theo partition (financial_ratio, price_board, company)")
if snapshot_df.empty:
    print("Chưa có snapshot partition.")
else:
    display(
        snapshot_df[
            [
                "category",
                "label",
                "partition_key",
                "partition",
                "status",
                "file_count",
                "total_rows",
                "total_size_human",
            ]
        ]
    )

# --- C) Master files ---
master_rows = []
for category, (file_path, label) in MASTER_FILES.items():
    exists = file_path.exists()
    master_rows.append(
        {
            "category": category,
            "label": label,
            "status": "OK" if exists else "MISSING",
            "path": str(file_path) if exists else "",
            "rows": _parquet_rows(file_path) if exists else 0,
            "size_human": _human_size(file_path.stat().st_size) if exists else _human_size(0),
            "last_modified": datetime.fromtimestamp(file_path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
            if exists
            else "",
        }
    )
master_df = pd.DataFrame(master_rows)
print("\nC) Master (ghi đè mỗi lần chạy)")
display(master_df)

# --- D) Run metadata (_runs) ---
runs_df = _read_run_metadata()
print("\nD) Metadata lần chạy OHLCV (_runs/<run_id>.json)")
if runs_df.empty:
    print("Chưa có file _runs — tạo khi gọi run_structure_ingestion_pipeline hoặc ghi metadata sau ingest.")
else:
    display(runs_df)

# --- E) Tổng dung lượng ---
def _dir_size(path: Path) -> int:
    if not path.exists():
        return 0
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += int(p.stat().st_size)
            except OSError:
                pass
    return total

size_by_category = []
for name in ["price", "index", "financial_ratio", "price_board", "company", "listing"]:
    cat_path = base / name
    if cat_path.exists():
        size_by_category.append({"category": name, "size_human": _human_size(_dir_size(cat_path))})

size_df = pd.DataFrame(size_by_category)
overall_bytes = _dir_size(base)
print("\nE) Dung lượng theo category + tổng Structure_Data")
if not size_df.empty:
    display(size_df)
print(f"Tổng raw/Structure_Data: {_human_size(overall_bytes)}")

0) Cấu hình ingest (cfg)
- data_lake_root: D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data
- run_date (metadata): 2026-05-18T171513
- end_date API: 2026-05-18
- tickers batch: 50 mã
- index batch: 5 mã
- incremental: False | window_days: 10
- years_back: 5 | sources: ['kbs', 'vci']
- watermark: {'price': {'last_trading_date': '2026-05-18', 'max_trading_date': '2026-05-18', 'run_id': '2026-05', 'updated_at_utc': '2026-05-18T08:55:53.526987+00:00'}, 'index': {'last_trading_date': '2026-05-18', 'max_trading_date': '2026-05-18', 'run_id': '2026-05', 'updated_at_utc': '2026-05-18T08:56:02.356654+00:00'}}

Gợi ý từ cell vừa chạy trong kernel (nếu có):


,category,session_var,symbols_or_files,parquet_paths
0,financial_ratio,ratio_outputs,50,0



A) OHLCV — layout year=*/month=*/<TICKER>.parquet (merge/dedupe theo tháng)


,category,label,status,symbol_count,expected_symbols,missing_symbols,file_count,month_partitions,partition_from,partition_to,trading_date_from,trading_date_to,total_rows,total_size_human
0,price,OHLCV cổ phiếu,OK,50,50,,3050,61,2021-05,2026-05,2021-05-18,2026-05-18,62340,25.38 MB
1,index,OHLCV chỉ số,OK,5,5,,305,61,2021-05,2026-05,2021-05-18,2026-05-18,6234,2.46 MB


A.1) Chi tiết theo mã (partition tháng + trading_date min/max)


...

,category,label,status,path,rows,size_human,last_modified
0,listing,Listing master,OK,D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\...,3253,82.61 KB,2026-05-18 14:34:28



D) Metadata lần chạy OHLCV (_runs/<run_id>.json)
Chưa có file _runs — tạo khi gọi run_structure_ingestion_pipeline hoặc ghi metadata sau ingest.

E) Dung lượng theo category + tổng Structure_Data


,category,size_human
0,price,25.38 MB
1,index,2.46 MB
2,financial_ratio,534.46 KB
3,price_board,29.23 KB
4,company,87.31 KB
5,listing,82.61 KB


Tổng raw/Structure_Data: 28.55 MB


: 